# ATLAS Solar Scenario Plotting Tutorial

This notebook creates maps from the future ATLAS scenario datasets produced by the scenario delta-correction workflow.

The workflow is designed for users who only need to edit the input parameters and run the notebook step by step.

The notebook can produce:

1. Monthly maps, by opening one NetCDF file for a selected month.
2. Annual mean maps, by opening all monthly files one by one, concatenating them along a `month` dimension, and then computing the annual average.

The expected input folder is the scenario ATLAS output created by the previous notebook:

`../data/atlas_data/{country}/{model}/{experiment}/`

Figures are saved here:

`../data/figures/{country}/{model}/{experiment}/`


## Step 1. Import Python libraries

Run this cell first. It loads the libraries used for reading NetCDF files and creating maps.


In [1]:
from pathlib import Path
import warnings

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

warnings.filterwarnings("ignore")

## Step 2. Set input parameters

Edit only this cell before running the notebook.

Set `annual_plot = True` to compute the annual mean. In that case, the notebook opens the monthly files one by one, concatenates them, and plots the average.

Set `annual_plot = False` to plot a single selected month.


In [2]:
# Country name used in folder and file names.
country = "peru"

# Solar variable used in the ATLAS workflow.
variable = "ssrd"

# Climate model and scenario experiment.
model = "CNRM-ESM2-1"
experiment = "ssp585"

# Future period used in the scenario files.
start_year = 2020
end_year = 2050

# Plot mode.
annual_plot = False

# Used only when annual_plot = False.
month = 1

# Input path: output from the scenario delta-correction notebook.
input_path = Path(f"../data/atlas_data/{country}/{model}/{experiment}/")

# Output path: folder where figures will be saved.
output_path = Path(f"../data/figures/{country}/{model}/{experiment}/")
output_path.mkdir(parents=True, exist_ok=True)

# File naming convention for the corrected scenario ATLAS files.
input_file_template = "ssrd_corrected_{country}_m{month}_{experiment}_{start_year}_{end_year}.nc"

# Variable names inside the NetCDF files.
corrected_variable_name = "ssrd_corrected"
delta_variable_name = "delta"

# Plot configuration.
corrected_vmin = 2
corrected_vmax = 9
corrected_cmap = "inferno_r"


delta_vmin = -100
delta_vmax = 100
delta_cmap = "bwr"

corrected_title = "Surface short-wave radiation downwards\n(kWh/m²/day)"
delta_title = "Scenario delta\n(kWh/m²/day)" 

## Step 3. Define helper functions

These functions open the monthly files, concatenate them when needed, and draw the maps.

No user edits are normally required in this section.


In [3]:
def find_required_file(folder: Path, filename: str) -> Path:
    """Return a required file path and raise a clear error if it is missing."""
    file_path = folder / filename
    if not file_path.exists():
        raise FileNotFoundError(
            f"Required file not found:\n{file_path}\n\n"
            "Check the input parameters and the folder structure."
        )
    return file_path


def scenario_file_for_month(month_number: int) -> Path:
    """Return the expected scenario ATLAS file path for one month."""
    filename = input_file_template.format(
        country=country,
        month=month_number,
        experiment=experiment,
        start_year=start_year,
        end_year=end_year,
    )
    return find_required_file(input_path, filename)


def open_monthly_dataset(month_number: int) -> xr.Dataset:
    """Open one monthly scenario ATLAS file and attach the month coordinate."""
    file_path = scenario_file_for_month(month_number)
    return xr.open_dataset(file_path).assign_coords(month=month_number)


def open_annual_mean_dataset(months=range(1, 13)) -> xr.Dataset:
    """Open monthly files one by one, concatenate them, and compute the annual mean."""
    monthly_datasets = []

    for month_number in months:
        ds = open_monthly_dataset(month_number)
        monthly_datasets.append(ds)

    combined = xr.concat(monthly_datasets, dim="month")
    annual_mean = combined.mean(dim="month", skipna=True)

    return annual_mean


def cell_edges(coord):
    """Compute grid cell edges from cell center coordinates."""
    coord = np.asarray(coord, dtype=float)
    d = np.diff(coord)

    edges = np.empty(coord.size + 1, dtype=float)
    edges[1:-1] = coord[:-1] + d / 2
    edges[0] = coord[0] - d[0] / 2
    edges[-1] = coord[-1] + d[-1] / 2

    return edges


def draw_map(
    data_array: xr.DataArray,
    vmin: float,
    vmax: float,
    cmap: str,
    title: str,
    label: str,
    fig_path: Path,
):
    """Draw and save a map from a latitude-longitude DataArray."""
    data_array = data_array.sortby("latitude")

    lon = data_array["longitude"].values
    lat = data_array["latitude"].values
    data = np.ma.masked_invalid(data_array.values)

    lon_b = cell_edges(lon)
    lat_b = cell_edges(lat)
    lon2d_b, lat2d_b = np.meshgrid(lon_b, lat_b)

    fig, ax = plt.subplots(
        figsize=(10, 12),
        subplot_kw={"projection": ccrs.PlateCarree()},
    )

    ax.add_feature(cfeature.OCEAN, facecolor="lightblue")
    ax.add_feature(cfeature.LAKES, edgecolor="black", facecolor="lightblue")
    ax.add_feature(cfeature.RIVERS, edgecolor="lightblue")
    ax.add_feature(cfeature.COASTLINE, edgecolor="black", linewidth=1.2)
    ax.add_feature(cfeature.BORDERS, edgecolor="black", linewidth=1.2)

    im = ax.pcolormesh(
        lon2d_b,
        lat2d_b,
        data,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        shading="flat",
        linewidth=0,
        antialiased=False,
    )

    cbar = fig.colorbar(
        im,
        ax=ax,
        orientation="vertical",
        shrink=0.75,
        aspect=25,
        pad=0.04,
    )
    cbar.set_label(label, fontsize=20)
    cbar.ax.tick_params(labelsize=20)

    gl = ax.gridlines(
        draw_labels=True,
        linewidth=0.5,
        color="gray",
        alpha=0.5,
        linestyle="--",
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    gl.xlabel_style = {"size": 20}
    gl.ylabel_style = {"size": 20}

    ax.set_extent(
        [lon.min() - 1, lon.max() + 1, lat.min() - 1, lat.max() + 1],
        crs=ccrs.PlateCarree(),
    )

    ax.set_title(title, fontsize=24)
    fig_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    return fig


def build_figure_path(prefix: str) -> Path:
    """Build the output figure path for either annual or monthly plots."""
    if annual_plot:
        filename = f"{prefix}_{country}_annual_{experiment}_{start_year}_{end_year}.png"
    else:
        filename = f"{prefix}_{country}_m{month}_{experiment}_{start_year}_{end_year}.png"

    return output_path / filename

## Step 4. Open the data to plot

Run this cell to open either the annual mean dataset or one selected monthly dataset.


In [4]:
if annual_plot:
    dataset_to_plot = open_annual_mean_dataset(months=range(1, 2))
else:
    dataset_to_plot = open_monthly_dataset(month)

In [5]:
if annual_plot == True:
    files_list = []
    for month in range(1,13):
        ref_path_template   = f'ssrd_integrated_{country}_m{month}.nc'
        ref_path   = f'../data/atlas_data/{country}/'
        ref_ds = xr.open_dataset(ref_path + ref_path_template).assign(month=month)
        files_list.append(ref_ds)
    xdf_all = xr.concat(files_list,dim='month')
    ref_ds = xdf_all.mean('month')
    
else:
    ref_path_template   = f'ssrd_integrated_{country}_m{month}.nc'
    ref_path   = f'../data/atlas_data/{country}/'
    ref_ds = xr.open_dataset(ref_path + ref_path_template)

In [6]:
# Opening the present data in order to express the delta in % with respect to the historical climatology:

delta_pct = (
    dataset_to_plot["delta"]
    / ref_ds["ssrd_integrated"]
) * 100

## Step 5. Plot the corrected future ATLAS dataset

This figure shows the future ATLAS solar radiation field after applying the scenario delta correction.


In [ ]:
corrected_fig_path = build_figure_path("ssrd_corrected")

draw_map(
    data_array=dataset_to_plot[corrected_variable_name],
    vmin=corrected_vmin,
    vmax=corrected_vmax,
    cmap=corrected_cmap,
    title="Surface Solar Radiation Downwards \n kWh/m²/day",#corrected_title,
    label="",
    fig_path=corrected_fig_path,
)

print(f"Saved: {corrected_fig_path}")

## Step 6. Plot the scenario delta

This figure shows the added scenario signal, computed as future downscaled scenario minus historical downscaled simulation.


In [ ]:
delta_fig_path = build_figure_path("delta_ssrd")

draw_map(
    data_array=delta_pct,#dataset_to_plot[delta_variable_name],
    vmin=-10,
    vmax=10,#delta_vmax,
    cmap=delta_cmap,
    title="Surface Solar Radiation Downwards \n % Anomaly",#corrected_title,
    label="",
    fig_path=delta_fig_path,
)

print(f"Saved: {delta_fig_path}")

## Step 7. Optional checks

Run this cell if you want to inspect the generated figure paths.


In [ ]:
print("Input folder:", input_path)
print("Output folder:", output_path)
print("Corrected figure:", corrected_fig_path)
print("Delta figure:", delta_fig_path)